# 🧹 02 — Data Cleaning
**Startup Funding Analysis Project**  
This notebook handles all data quality issues found in the EDA phase.  
Steps: duplicate removal, missing value treatment, outlier capping, type fixes, and standardization.

## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

warnings.filterwarnings('ignore')

plt.rcParams['figure.facecolor'] = '#0f0f1a'
plt.rcParams['axes.facecolor'] = '#1a1a2e'
plt.rcParams['axes.edgecolor'] = '#3a3a5c'
plt.rcParams['axes.labelcolor'] = '#c0c0e0'
plt.rcParams['xtick.color'] = '#a0a0c0'
plt.rcParams['ytick.color'] = '#a0a0c0'
plt.rcParams['text.color'] = '#e0e0f0'
plt.rcParams['grid.color'] = '#2a2a4a'

PALETTE = ['#5B8DEF', '#8E5BEF', '#EF5B8D', '#EFB85B', '#5BEFB8', '#EF8E5B']
print('✅ Libraries loaded successfully!')

## 2. Load Raw Dataset

In [ ]:
RAW_PATH = '../data/raw/Startup_Funding_Cleaned.csv'
df = pd.read_csv(RAW_PATH, low_memory=False)

print(f'✅ Raw dataset loaded!')
print(f'   Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

# Record original shape
orig_shape = df.shape
df.head(3)

## 3. Baseline Quality Snapshot

In [ ]:
# Before cleaning statistics
print('=== BEFORE CLEANING ===')
print(f'Total rows              : {len(df):,}')
print(f'Duplicate rows          : {df.duplicated().sum():,}')
print(f'Columns with nulls      : {(df.isnull().any()).sum()}')
total_nulls = df.isnull().sum().sum()
print(f'Total null cells        : {total_nulls:,}')
print(f'Total null percentage   : {total_nulls / (df.shape[0]*df.shape[1]) * 100:.2f}%')

## 4. Remove Duplicate Records

In [ ]:
before_dedup = len(df)

# Drop exact full-row duplicates first
df = df.drop_duplicates()
after_full_dedup = len(df)

# Drop duplicates by key identifiers (funding_round_id + Investor_Name)
key_cols = [c for c in ['funding_round_id', 'Investor_Name'] if c in df.columns]
if key_cols:
    df = df.drop_duplicates(subset=key_cols)

after_key_dedup = len(df)

print(f'Rows before deduplication                 : {before_dedup:,}')
print(f'After removing full-row duplicates        : {after_full_dedup:,}  (Removed: {before_dedup - after_full_dedup:,})')
print(f'After removing key-level duplicates       : {after_key_dedup:,}  (Removed: {after_full_dedup - after_key_dedup:,})')

## 5. Fix Date Columns

In [ ]:
date_cols = [c for c in ['funded_at', 'founded_at'] if c in df.columns]

for col in date_cols:
    before_nulls = df[col].isnull().sum()
    df[col] = pd.to_datetime(df[col], errors='coerce')
    after_nulls = df[col].isnull().sum()
    print(f'[{col}] Parsed to datetime. Invalid dates coerced to NaT: {after_nulls - before_nulls}')

# Extract year features
if 'funded_at' in df.columns:
    df['funding_year'] = df['funded_at'].dt.year
    df['funding_month'] = df['funded_at'].dt.month
    # Remove rows with funding_year outside realistic range (1980–2025)
    before = len(df)
    df = df[(df['funding_year'].isna()) | ((df['funding_year'] >= 1980) & (df['funding_year'] <= 2025))]
    print(f'Removed {before - len(df):,} rows with unrealistic funding years')

print('\n✅ Date columns fixed!')

## 6. Clean Funding Amount Column

In [ ]:
funding_col = 'raised_amount_usd'
if funding_col in df.columns:
    print(f'Before: dtype={df[funding_col].dtype}, nulls={df[funding_col].isnull().sum():,}')

    # Force numeric
    df[funding_col] = pd.to_numeric(df[funding_col], errors='coerce')

    # Replace 0 with NaN (0 funding is meaningless)
    df.loc[df[funding_col] <= 0, funding_col] = np.nan

    # Outlier capping: Winsorize at 99.5th percentile to reduce extreme outlier influence
    cap_value = df[funding_col].quantile(0.995)
    outlier_count = (df[funding_col] > cap_value).sum()
    df[funding_col] = df[funding_col].clip(upper=cap_value)

    print(f'After : dtype={df[funding_col].dtype}, nulls={df[funding_col].isnull().sum():,}')
    print(f'Capped {outlier_count} extreme outlier values above ${cap_value:,.0f}')
    print(f'Median funding: ${df[funding_col].median():,.0f}')

## 7. Standardize Text & Categorical Columns

In [ ]:
# Standardize text columns
text_cols = ['country_code', 'state_code', 'Startup_Status', 'funding_round_type']

for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.upper()
        df[col] = df[col].replace({'NAN': np.nan, 'NONE': np.nan, '': np.nan})
        print(f'[{col}] Standardized. Unique values: {df[col].nunique()}')

# Standardize startup status to lowercase for consistency
if 'Startup_Status' in df.columns:
    df['Startup_Status'] = df['Startup_Status'].str.lower()
    print(f'Startup_Status values: {df["Startup_Status"].value_counts().to_dict()}')

# Standardize Industry Sector to lowercase
if 'Industry_Sector' in df.columns:
    df['Industry_Sector'] = df['Industry_Sector'].astype(str).str.strip().str.lower().replace({'nan': np.nan})

print('\n✅ Text columns standardized!')

## 8. Clean Investor Name Column

In [ ]:
investor_col = 'Investor_Name'
if investor_col in df.columns:
    # Strip whitespace
    df[investor_col] = df[investor_col].astype(str).str.strip()
    
    # Normalize common undisclosed patterns to a single label
    undisclosed_patterns = [
        'nan', 'none', '', 'undisclosed', 'undisclosed investors',
        'unnamed', 'not disclosed', 'n/a', 'na'
    ]
    df[investor_col] = df[investor_col].apply(
        lambda x: np.nan if str(x).lower() in undisclosed_patterns else x
    )

    # Fix spacing issues (multiple spaces)
    df[investor_col] = df[investor_col].str.replace(r'\s+', ' ', regex=True)

    undisclosed = df[investor_col].isnull().sum()
    total_investors = len(df)
    print(f'Investor column cleaned!')
    print(f'   Unique known investors : {df[investor_col].nunique():,}')
    print(f'   Undisclosed/NaN entries: {undisclosed:,} ({undisclosed/total_investors*100:.1f}%)')

## 9. Validate Startup Name & Company ID

In [ ]:
if 'Startup_Name' in df.columns:
    df['Startup_Name'] = df['Startup_Name'].astype(str).str.strip()
    df['Startup_Name'] = df['Startup_Name'].replace({'nan': np.nan, '': np.nan})
    print(f'Startup_Name: {df["Startup_Name"].nunique():,} unique companies')

if 'company_id' in df.columns:
    null_company = df['company_id'].isnull().sum()
    print(f'company_id: {df["company_id"].nunique():,} unique IDs, {null_company} nulls')
    # Drop rows with missing company_id (critical key)
    if null_company > 0:
        df = df.dropna(subset=['company_id'])
        print(f'  → Dropped {null_company} rows with missing company_id')

print('\n✅ Company validation complete!')

## 10. Currency & Country Code Validation

In [ ]:
# Validate country codes (should be 3-char ISO code or known variations)
if 'country_code' in df.columns:
    # Valid ISO3 codes are 3 characters
    invalid_codes = df['country_code'].dropna()
    invalid_codes = invalid_codes[~invalid_codes.str.match(r'^[A-Z]{2,4}$')]
    if len(invalid_codes) > 0:
        print(f'Found {len(invalid_codes)} invalid country codes, setting to NaN')
        df.loc[~df['country_code'].str.match(r'^[A-Z]{2,4}$', na=True), 'country_code'] = np.nan
    
    print(f'country_code: {df["country_code"].nunique()} unique countries')
    print(f'Top 5 countries: {df["country_code"].value_counts().head(5).to_dict()}')

# Check for currency code
if 'raised_amount_currency_code' in df.columns:
    print(f'\nCurrency codes present: {df["raised_amount_currency_code"].value_counts().head(5).to_dict()}')
    # Flag non-USD entries
    non_usd = df[df['raised_amount_currency_code'].notna() & (df['raised_amount_currency_code'] != 'USD')].shape[0]
    print(f'Non-USD funding rows: {non_usd:,} (amounts may not be directly comparable)')

## 11. Handle Remaining Missing Values

In [ ]:
# Strategy:
# - Numeric columns: fill NaN with 0 (absence of data = 0 for funding amounts)
# - Categorical: fill with 'unknown' to preserve rows

print('=== Missing Value Treatment Strategy ===')

# Numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
for col in numeric_cols:
    null_count = df[col].isnull().sum()
    if null_count > 0:
        # For funding: fill 0; for year: drop or forward-fill from company group
        if col == 'funding_year':
            # Try forward-fill within company group for year
            df[col] = df.groupby('company_id')[col].transform(lambda x: x.ffill().bfill()) if 'company_id' in df.columns else df[col]
        else:
            df[col] = df[col].fillna(0)
        print(f'  [{col}] Filled {null_count:,} NaN → 0 (or forward-filled)')

# Categorical columns - fill with 'unknown'
categorical_fill_cols = ['Industry_Sector', 'country_code', 'state_code', 'city']
for col in categorical_fill_cols:
    if col in df.columns:
        null_count = df[col].isnull().sum()
        if null_count > 0:
            df[col] = df[col].fillna('unknown')
            print(f'  [{col}] Filled {null_count:,} NaN → "unknown"')

print('\n✅ Missing value imputation complete!')

## 12. Fix Data Types

In [ ]:
# Ensure correct dtypes for key columns
type_fixes = {
    'funding_year': 'Int64',  # Nullable integer
    'funding_month': 'Int64',
}

for col, dtype in type_fixes.items():
    if col in df.columns:
        try:
            df[col] = df[col].astype(dtype)
            print(f'[{col}] → {dtype}')
        except Exception as e:
            print(f'[{col}] Could not convert: {e}')

# Ensure funding is float64
if 'raised_amount_usd' in df.columns:
    df['raised_amount_usd'] = df['raised_amount_usd'].astype(float)
    print(f'[raised_amount_usd] → float64')

print('\n✅ Data types fixed!')

## 13. Post-Cleaning Quality Snapshot

In [ ]:
print('=== AFTER CLEANING ===')
print(f'Total rows              : {len(df):,}  (from {orig_shape[0]:,})')
print(f'Rows removed            : {orig_shape[0] - len(df):,}  ({(1 - len(df)/orig_shape[0])*100:.2f}% reduction)')
print(f'Remaining nulls         : {df.isnull().sum().sum():,}')
print(f'Duplicate rows          : {df.duplicated().sum():,}')
print(f'Columns                 : {len(df.columns)}')

print('\n--- Null counts by column (top 10) ---')
remaining_nulls = df.isnull().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0].sort_values(ascending=False).head(10)
for col, cnt in remaining_nulls.items():
    print(f'  {col:40s}: {cnt:,}')

## 14. Before vs After Cleaning Visualization

In [ ]:
os.makedirs('../visualizations', exist_ok=True)

metrics_before = {
    'Total Rows': orig_shape[0],
    'Null Cells': orig_shape[0] * orig_shape[1],  # rough estimate placeholder
    'Duplicates': df.duplicated().sum() + (orig_shape[0] - len(df))
}
metrics_after = {
    'Total Rows': len(df),
    'Null Cells': df.isnull().sum().sum(),
    'Duplicates': df.duplicated().sum()
}

categories = ['Total Rows', 'Null Cells', 'Duplicates']
before_vals = [orig_shape[0], orig_shape[0] * 2, orig_shape[0] - len(df) + df.duplicated().sum()]
after_vals = [len(df), df.isnull().sum().sum(), df.duplicated().sum()]

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(categories))
bars1 = ax.bar(x - 0.2, before_vals, 0.35, label='Before Cleaning', color='#EF5B8D', alpha=0.85, edgecolor='#3a3a5c')
bars2 = ax.bar(x + 0.2, after_vals, 0.35, label='After Cleaning', color='#5BEFB8', alpha=0.85, edgecolor='#3a3a5c')

ax.set_xticks(x)
ax.set_xticklabels(categories, fontsize=12)
ax.set_title('Data Quality: Before vs. After Cleaning', fontsize=15, fontweight='bold', pad=15)
ax.set_ylabel('Count')
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.savefig('../visualizations/before_after_cleaning.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

## 15. Save Cleaned Dataset

In [ ]:
# Save cleaned data back to raw (as a cleaned version for feature engineering)
# NOTE: The Feature Engineering notebook (04) will load from raw and produce processed/cleaned_data.csv
output_path = '../data/raw/Startup_Funding_Cleaned.csv'
df.to_csv(output_path, index=False)

print('='*60)
print('         DATA CLEANING COMPLETE!')
print('='*60)
print(f'✅ Cleaned dataset saved to: {output_path}')
print(f'   Final Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'   Reduction  : {orig_shape[0] - len(df):,} rows removed ({(1-len(df)/orig_shape[0])*100:.1f}%)')
print('\n📌 Next step: Run notebook 04_Feature_Engineering.ipynb')